# 01 – Adatfelderítés: ASVspoof 2019 LA

Protokoll betöltése, címke- és rendszereloszlás, egy bonafide és egy A01 (spoof) minta mel-spektrogramja.

In [ ]:
import sys
from pathlib import Path

import yaml
import librosa.display
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from features import load_audio, mel_db
from dataset import load_protocol

cfg = yaml.safe_load(open(PROJECT_ROOT / "config.yaml"))
A = cfg["audio"]

# Kaggle-ön cseréld pl.: DATA_ROOT = Path("/kaggle/input/asvpoof-2019-dataset/LA/LA")
DATA_ROOT = PROJECT_ROOT / cfg["data"]["root"]
DATA_ROOT

In [ ]:
df = load_protocol(DATA_ROOT, "train")
print(len(df), "sor")
df.head()

In [ ]:
print(df["label"].value_counts(), "\n")
print(df["system"].value_counts())

In [ ]:
bona = df[df["label"] == "bonafide"].iloc[0]
a01 = df[df["system"] == "A01"].iloc[0]
bona["file"], a01["file"]

In [ ]:
def spec(path):
    y = load_audio(path, sr=A["sr"], max_sec=A["max_sec"])
    return mel_db(y, sr=A["sr"], n_fft=A["n_fft"], hop=A["hop"], n_mels=A["n_mels"])

S_bona = spec(bona["path"])
S_a01 = spec(a01["path"])
S_bona.shape, S_a01.shape

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for ax, S, title in zip(axes, [S_bona, S_a01], [f"bonafide – {bona['file']}", f"A01 spoof – {a01['file']}"]):
    img = librosa.display.specshow(S, sr=A["sr"], hop_length=A["hop"], x_axis="time", y_axis="mel", ax=ax)
    ax.set_title(title)
fig.colorbar(img, ax=axes, format="%+2.0f dB")
plt.show()